In [ ]:
import os, queue, threading, contextlib, sounddevice as sd
from dotenv import load_dotenv
from deepgram import DeepgramClient

load_dotenv()
dg = DeepgramClient(api_key=os.environ["DEEPGRAM_API_KEY"])
RATE = int(sd.query_devices(kind="input")["default_samplerate"])

# Names, jargon, product terms. Biggest accuracy lever of anything tested here -
# on a hard sentence this took flux from 9.1% to 0% word error, nova-3 from 13.6% to 4.5%.
KEYTERMS = ["Kubernetes", "Priya", "Postgres", "standup"]

OUT = sd.RawOutputStream(samplerate=24000, channels=1, dtype="int16", latency="low")
OUT.start()

_tail = b""
def play(chunk):
    global _tail                     # chunk boundaries don't respect 16-bit frames
    buf = _tail + chunk
    cut = len(buf) - len(buf) % 2
    OUT.write(buf[:cut])
    _tail = buf[cut:]

RATE

In [ ]:
# STT 1 - nova-3, waits for a pause before showing anything. Accurate, ~2.1s behind you.
PAUSE_MS = 800

q = queue.Queue()
with dg.listen.v1.connect(
    model="nova-3", encoding="linear16", sample_rate=RATE, channels=1,
    interim_results=True, punctuate=True, smart_format=True,
    endpointing=PAUSE_MS, utterance_end_ms=1000,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        line = []
        try:
            for m in sock:
                if m.type != "Results":
                    continue
                text = m.channel.alternatives[0].transcript
                if m.is_final:
                    if text:
                        line.append(text)
                    if m.speech_final and line:
                        print(" ".join(line).ljust(100))
                        line = []
                elif text:
                    print(" ".join(line + [text])[:100].ljust(100), end="\r", flush=True)
        except KeyboardInterrupt:
            q.put(None)

In [ ]:
# STT 2 - flux, predicts when your turn ends. Fast, but loose on names/jargon
# unless you give it keyterms (see STT 4).
EAGER, EOT = 0.3, 0.5

q = queue.Queue()
with dg.listen.v2.connect(
    model="flux-general-en", encoding="linear16", sample_rate=RATE,
    eager_eot_threshold=EAGER, eot_threshold=EOT, eot_timeout_ms=2000,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        try:
            for m in sock:
                ev = getattr(m, "event", None)
                if ev == "EndOfTurn":
                    print(m.transcript.ljust(100))
                elif ev in ("StartOfTurn", "Update", "EagerEndOfTurn") and m.transcript:
                    print(m.transcript[:100].ljust(100), end="\r", flush=True)
        except KeyboardInterrupt:
            q.put(None)

In [ ]:
# STT 3 - nova-3 accuracy at roughly half the wait (0.80s vs 2.08s measured).
# Two separate signals instead of one: each phrase prints the moment it is final,
# and the line only breaks on UtteranceEnd - a real pause, not just a short gap.
PAUSE_MS = 150     # how fast a phrase finalises (lower = text lands sooner)
BREAK_MS = 1000    # silence that counts as a conversational break -> new line

q = queue.Queue()
with dg.listen.v1.connect(
    model="nova-3", encoding="linear16", sample_rate=RATE, channels=1,
    interim_results=True, punctuate=True, smart_format=True,
    endpointing=PAUSE_MS, utterance_end_ms=BREAK_MS, keyterm=KEYTERMS,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        dirty = False
        try:
            for m in sock:
                if m.type == "UtteranceEnd":
                    if dirty:
                        print()
                        dirty = False
                elif m.type == "Results" and m.is_final:
                    text = m.channel.alternatives[0].transcript
                    if text:
                        print(text + " ", end="", flush=True)
                        dirty = True
        except KeyboardInterrupt:
            q.put(None)

In [ ]:
# STT 4 - flux with keyterms. Most accurate of the four (0% word error on the test
# sentence vs 9.1% without them), ~1.9s behind you. Add your own vocabulary to KEYTERMS.
EAGER, EOT = 0.3, 0.5

q = queue.Queue()
with dg.listen.v2.connect(
    model="flux-general-en", encoding="linear16", sample_rate=RATE,
    eager_eot_threshold=EAGER, eot_threshold=EOT, eot_timeout_ms=2000,
    keyterm=KEYTERMS,
) as sock:
    threading.Thread(target=lambda: [sock.send_media(b) for b in iter(q.get, None)], daemon=True).start()
    with sd.RawInputStream(samplerate=RATE, channels=1, dtype="int16", blocksize=RATE // 20,
                           callback=lambda data, *_: q.put(bytes(data))):
        print("listening - interrupt the cell to stop\n")
        try:
            for m in sock:
                ev = getattr(m, "event", None)
                if ev == "EndOfTurn":
                    print(m.transcript.ljust(100))
                elif ev in ("StartOfTurn", "Update", "EagerEndOfTurn") and m.transcript:
                    print(m.transcript[:100].ljust(100), end="\r", flush=True)
        except KeyboardInterrupt:
            q.put(None)

In [ ]:
# TTS 1 - one-shot REST, plays chunks as they arrive. ~0.84s to first sound,
# most of which is setting up the connection.
def say(text, model="aura-2-thalia-en"):
    for chunk in dg.speak.v1.audio.generate(
            text=text, model=model, encoding="linear16",
            container="none",        # default is wav - the header clicks through the speakers
            sample_rate=24000):
        play(chunk)

say("Hey Marcus, this is the one shot version.")

In [ ]:
# TTS 2 - websocket, new connection each call. Takes text in pieces, so you can
# feed LLM tokens. ~0.80s to first sound.
from deepgram.speak.v1.types import SpeakV1Text

def say_stream(pieces, model="aura-2-thalia-en"):
    with dg.speak.v1.connect(model=model, encoding="linear16", sample_rate=24000) as sock:
        def feed():
            for p in pieces:
                sock.send_text(SpeakV1Text(type="Speak", text=p))
            sock.send_flush()
        threading.Thread(target=feed, daemon=True).start()
        for m in sock:
            if isinstance(m, bytes):
                play(m)
            elif getattr(m, "type", None) == "Flushed":
                break

say_stream(w + " " for w in "This text is arriving one word at a time.".split())

In [ ]:
# TTS 3 - the startup fix. Nearly all of that 0.8s was the handshake, not the model,
# so hold the socket open: 0.22s per line, measured on your key. Run this cell once.
SPEAKER = contextlib.ExitStack()
SOCK = SPEAKER.enter_context(
    dg.speak.v1.connect(model="aura-2-thalia-en", encoding="linear16", sample_rate=24000))
MUTED = False

def _pump():
    try:
        for m in SOCK:
            if isinstance(m, bytes) and not MUTED:
                play(m)
    except Exception:
        pass                          # socket closed
threading.Thread(target=_pump, daemon=True).start()

def say_now(text):
    global MUTED
    MUTED = False
    SOCK.send_text(SpeakV1Text(type="Speak", text=text))
    SOCK.send_flush()

def stop():
    global MUTED                      # don't OUT.abort() here - killing the stream from
    MUTED = True                      # another thread takes PortAudio down with it
    SOCK.send_clear()

say_now("This starts almost instantly, because the connection is already open.")

In [ ]:
# TTS 4 - same open socket, fed a piece at a time: lowest latency AND token streaming.
# This is the one to use behind an LLM. stop() from TTS 3 works here too for barge-in.
def say_tokens(pieces):
    global MUTED
    MUTED = False
    for p in pieces:
        SOCK.send_text(SpeakV1Text(type="Speak", text=p))
    SOCK.send_flush()

say_tokens(w + " " for w in
           "Streaming text in as it is generated means the first word plays before the last one exists.".split())